In [18]:
from datetime import date
from pathlib import Path

import blpapi
import pandas as pd

END_DATE = date.today()
START_DATE = END_DATE.replace(year=END_DATE.year - 10)
FIELD = ["PX_LAST"]
DATA_DIR = Path("data")

UNIVERSES = {
    "fx": {
        "EURUSD": "EURUSD Curncy",
        "USDJPY": "USDJPY Curncy",
        "GBPUSD": "GBPUSD Curncy",
        "USDCHF": "USDCHF Curncy",
        "USDCAD": "USDCAD Curncy",
        "AUDUSD": "AUDUSD Curncy",
        "NZDUSD": "NZDUSD Curncy",
        "USDNOK": "USDNOK Curncy",
        "USDSEK": "USDSEK Curncy",
    },
    "equities": {
        "SP500": "SPX Index",
        "NASDAQ100": "NDX Index",
        "DOW": "INDU Index",
        "RUSSELL2000": "RTY Index",
        "NASDAQ_COMPOSITE": "CCMP Index",
    },
    "treasuries": {
        "1M": "USGG1M Index",
        "3M": "USGG3M Index",
        "6M": "USGG6M Index",
        "1Y": "USGG12M Index",
        "2Y": "USGG2YR Index",
        "5Y": "USGG5YR Index",
        "10Y": "USGG10YR Index",
        "30Y": "USGG30YR Index",
    },
    "commodities": {
        "WTI": "CL1 Comdty",
        "BRENT": "CO1 Comdty",
        "GOLD": "GC1 Comdty",
        "SILVER": "SI1 Comdty",
        "COPPER": "HG1 Comdty",
        "CORN": "C 1 Comdty",
        "SOYBEANS": "S 1 Comdty",
        "WHEAT": "W 1 Comdty",
        "LIVE_CATTLE": "LC1 Comdty",
        "COTTON": "CT1 Comdty",
        "COFFEE": "KC1 Comdty",
        "SUGAR": "SB1 Comdty",
    },
}

TENORS = UNIVERSES["treasuries"]
SECURITY = list(TENORS.values())
CURVE_DATES = [
    START_DATE,
    START_DATE + (END_DATE - START_DATE) // 2,
    END_DATE,
]


def _element_value(row, field):
    if not row.hasElement(field) or row.getElement(field).isNull():
        return None
    return row.getElementAsFloat(field)


def fetch_category(category, universe, fields=FIELD, start_date=START_DATE, end_date=END_DATE):
    """Pull daily Bloomberg history for one configurable instrument universe."""
    if not universe:
        raise ValueError(f"{category} universe must contain at least one security")
    if not isinstance(start_date, date) or not isinstance(end_date, date):
        raise TypeError("start_date and end_date must be datetime.date objects")
    if start_date > end_date:
        raise ValueError("start_date must be on or before end_date")

    security_names = {security: name for name, security in universe.items()}
    session = blpapi.Session()
    if not session.start():
        raise RuntimeError("Could not start Bloomberg session. Is Bloomberg Terminal running?")

    try:
        if not session.openService("//blp/refdata"):
            raise RuntimeError("Could not open Bloomberg reference-data service.")

        request = session.getService("//blp/refdata").createRequest("HistoricalDataRequest")
        for security in universe.values():
            request.getElement("securities").appendValue(security)
        for field in fields:
            request.getElement("fields").appendValue(field)
        request.set("startDate", start_date.strftime("%Y%m%d"))
        request.set("endDate", end_date.strftime("%Y%m%d"))
        request.set("periodicitySelection", "DAILY")
        session.sendRequest(request)

        records = []
        while True:
            event = session.nextEvent()
            for message in event:
                if not message.hasElement("securityData"):
                    continue
                security_data = message.getElement("securityData")
                security = security_data.getElementAsString("security")
                field_data = security_data.getElement("fieldData")
                for index in range(field_data.numValues()):
                    row = field_data.getValueAsElement(index)
                    record = {
                        "category": category,
                        "name": security_names.get(security, security),
                        "security": security,
                        "date": row.getElementAsDatetime("date"),
                    }
                    record.update({field.lower(): _element_value(row, field) for field in fields})
                    records.append(record)
            if event.eventType() == blpapi.Event.RESPONSE:
                break
    finally:
        session.stop()

    columns = ["category", "name", "security", "date", *[field.lower() for field in fields]]
    return pd.DataFrame(records, columns=columns).sort_values(["security", "date"])


datasets = {}
for category, universe in UNIVERSES.items():
    dataset = fetch_category(category, universe)
    datasets[category] = dataset
    category_dir = DATA_DIR / category
    category_dir.mkdir(parents=True, exist_ok=True)
    output_path = category_dir / f"{category}.csv"
    dataset.to_csv(output_path, index=False)
    print(f"{category}: saved {len(dataset):,} rows across {len(universe)} securities to {output_path}")

prices = datasets["treasuries"].set_index(["security", "date"])[["px_last"]]


fx: saved 23,480 rows across 9 securities to data\fx\fx.csv
equities: saved 12,565 rows across 5 securities to data\equities\equities.csv
treasuries: saved 20,860 rows across 8 securities to data\treasuries\treasuries.csv
commodities: saved 30,266 rows across 12 securities to data\commodities\commodities.csv


In [19]:
from IPython.display import HTML, display
import plotly.express as px

plot_data = prices.reset_index()
plot_data["tenor"] = plot_data["security"].map({security: tenor for tenor, security in TENORS.items()})
plot_data["date"] = pd.to_datetime(plot_data["date"])
plot_data["tenor_order"] = plot_data["tenor"].map({tenor: index for index, tenor in enumerate(TENORS)})
plot_data = plot_data.sort_values(["date", "tenor_order"])

history_figure = px.line(
    plot_data,
    x="date",
    y="px_last",
    color="tenor",
    markers=True,
    category_orders={"tenor": list(TENORS)},
    title="US Treasury Yields Through Time",
    labels={"px_last": "Yield / PX_LAST", "date": "Date", "tenor": "Tenor"},
)
display(HTML(history_figure.to_html(include_plotlyjs="cdn", full_html=False)))

available_dates = plot_data["date"].dt.normalize().drop_duplicates().sort_values().tolist()
snapshot_rows = []
for target_date in CURVE_DATES:
    nearest_date = min(available_dates, key=lambda value: abs(value - pd.Timestamp(target_date)))
    snapshot_rows.append(
        plot_data[plot_data["date"].dt.normalize() == nearest_date].assign(
            snapshot_date=nearest_date.date()
        )
    )

snapshots = pd.concat(snapshot_rows, ignore_index=True)
curve_figure = px.line(
    snapshots,
    x="tenor",
    y="px_last",
    color="snapshot_date",
    markers=True,
    category_orders={"tenor": list(TENORS)},
    title="US Treasury Yield Curve Shape",
    labels={"px_last": "Yield / PX_LAST", "tenor": "Tenor", "snapshot_date": "Snapshot date"},
)
display(HTML(curve_figure.to_html(include_plotlyjs=False, full_html=False)))